In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import yaml
from ml_collections.config_dict import ConfigDict

import torch
from sequence_generation.utils import load_seed, load_dataloader, load_generator, load_regressor, expand_simplex, sample_cond_prob_path
from sequence_generation.solver import SimplexEulerSolver

In [3]:
if torch.cuda.is_available():
    device = 'cuda:0'
    print('Using gpu')
else:
    device = 'cpu'
    print('Using cpu.')

Using gpu


In [4]:
load_seed(42)

# Load dataset/model

In [5]:
config_file_name = "/nfs/team361/dj16/projects/sequence_generation/configs/enhancer_gosai.yaml"
config_file = yaml.load(open(config_file_name, "r"), yaml.FullLoader)
config = ConfigDict(config_file)

In [6]:
dirichlet_fm_model, optimizer, lr_scheduler = load_generator(config)

In [7]:
train_loader, val_loader, test_loader = load_dataloader(config)

/nfs/team361/dj16/pypoetry/virtualenvs/sequence-generation-7Ds7Y9Ey-py3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


# Generate sequence 

In [8]:
# Load data
for i, batch in enumerate(train_loader):
    print(batch)
    break

{'seqs': tensor([[2, 2, 0,  ..., 1, 3, 2],
        [1, 1, 0,  ..., 3, 2, 2],
        [0, 1, 1,  ..., 0, 0, 1],
        ...,
        [1, 3, 0,  ..., 2, 0, 3],
        [1, 0, 1,  ..., 3, 1, 3],
        [2, 3, 3,  ..., 1, 3, 0]]), 'clss': tensor([[-0.5299, -0.5612, -1.0305],
        [-0.2805, -1.0700,  0.5337],
        [ 1.0994,  0.4908,  0.2275],
        ...,
        [ 3.6346,  2.4917,  2.6065],
        [ 0.1279,  0.7716, -0.3097],
        [-0.4896, -0.1689, -0.7780]], dtype=torch.float64), 'attention_mask': tensor([[1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        ...,
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.]])}


In [9]:
# Expand discrete variable into consinous simplex
xt, alphas = sample_cond_prob_path(config.model, batch['seqs'], config.model.alphabet_size)
prior_pseudocount = 0.1
xt_inp, prior_weights = expand_simplex(xt, alphas, prior_pseudocount)

In [10]:
xt_inp.shape

torch.Size([512, 200, 8])

In [11]:
# Run model: get flow
logits = dirichlet_fm_model(seq=xt_inp, t=alphas)

In [ ]:
# Define solver
N = 6
T = torch.linspace(0, 1, N)  # sample times
T = T.to(device=device)

solver = SimplexEulerSolver(config, dirichlet_fm_model)
B, L, _ = xt_inp.shape 
_, _, seq_pred = solver.sample(B,L)

In [1]:
sol

NameError: name 'sol' is not defined